# Ethanol supply and sugar diversion

This notebook builds one table by ethanol supply year (ESY). It shows how much ethanol came from the sugar side (cane juice and syrup, B-heavy and C-heavy molasses) against grains and maize, the price paid for cane-juice ethanol, and how much sugar was diverted.

ESY runs roughly November to October, so it lines up closely with the sugar season (October to September), but not exactly. I treat an ESY as the matching sugar season, and I say so in the README too.

In [1]:
import re
import pandas as pd
import pdfplumber

RAW = "data/raw/"

In [2]:
# the supply table in this reply is real text, so I read it straight from the PDF
with pdfplumber.open(RAW + "rs_ethanol_supply_2026.pdf") as pdf:
    text = " ".join(page.extract_text() or "" for page in pdf.pages)
text = " ".join(text.split())

In [3]:
years = ["2020-21", "2021-22", "2022-23", "2023-24", "2024-25", "2025-26"]
# in the PDF the label wraps around the numbers, so each pattern only needs its first line
rows = {
    "juice_syrup": r"Sugarcane Juice/(?: Sugar/Sugar Syrup)?",
    "b_heavy": r"B- ?Heavy Molasses",
    "c_heavy": r"C- ?Heavy Molasses",
    "damaged_grain": r"Damaged Food(?: Grains)?",
    "fci_rice": r"Surplus FCI Rice",
    "maize": r"Maize",
    "total": r"Total Receipt \(Cr(?: Lit\))?",
}
six = r"\s+((?:[\d.]+\s+){5}[\d.]+)"


def grab(label, nth=0):
    found = re.findall(label + six, text)
    if len(found) <= nth:
        raise ValueError("could not find " + label)
    return [float(x) for x in found[nth].split()]


# first match of each label is the supply table (crore litres), the second sugar syrup match is the price table
eth = pd.DataFrame({k: grab(v) for k, v in rows.items()}, index=years)
eth["juice_price"] = grab(rows["juice_syrup"], 1)

The feedstocks should add up to the printed total for every year, so this gap should be 0.

In [4]:
# the feedstocks should add up to the printed total in every year
gap = (eth[list(rows)[:-1]].sum(axis=1) - eth["total"]).abs().max()
gap

eth["sugar_route"] = eth[["juice_syrup", "b_heavy", "c_heavy"]].sum(axis=1)
eth["sugar_route_share"] = (eth["sugar_route"] / eth["total"] * 100).round(1)

The diversion table in the second reply is an image, so I typed its four years into `rs_diversion_2026_typed.csv`.

In [5]:
# the diversion table is an image in the PDF, so these four years are typed from it
div = pd.read_csv(RAW + "rs_diversion_2026_typed.csv", index_col="esy")
eth = eth.join(div)
eth.index.name = "esy"

In [6]:
eth.to_csv("data/clean/ethanol.csv")
eth

,juice_syrup,b_heavy,c_heavy,damaged_grain,fci_rice,maize,total,juice_price,sugar_route,sugar_route_share,diversion_lmt,maize_lmt
esy,,,,,,,,,,,,
2020-21,38.16,178.67,38.07,38.46,2.19,0.00,295.55,62.65,254.90,86.2,NaN,NaN
2021-22,84.98,253.34,10.21,23.49,46.72,0.00,418.74,63.45,348.53,83.2,NaN,NaN
2022-23,128.35,235.33,5.61,31.90,73.71,31.51,506.41,65.61,369.29,72.9,43.0,8.3
2023-24,65.51,149.10,57.79,116.56,0.13,289.91,679.00,65.61,272.40,40.1,24.0,75.4
2024-25,165.31,138.81,16.90,80.48,140.09,498.49,1040.08,65.61,321.02,30.9,35.0,131.1
2025-26,143.95,81.88,11.80,44.98,176.77,257.91,717.29,65.61,237.63,33.1,28.0,67.9
